# AMC 2026 – Round 3 · Notebook 1: the Bloomberg data (BABQM)
Matterhorn Capital Management · Group 3 · built on the notebook of TA session 6

**What this notebook does.** It reads the Bloomberg data of our quant strategy, the EQS screen **"AMC R3 BABQM"** (low beta + quality + momentum, top 2% of the Russell 1000), cleans and checks it, and saves the CSV files for Notebook 2.

It has two parts:
1. **Backtests and trials log (§1).** Our EQBT exports and our list of tested strategies are already in `../Data/`, so this part runs now.
2. **Today's stocks and 60 months of prices (§2–§3).** They come from `Bloomberg template/r3_bloomberg.xlsx`, refreshed on the terminal on **Saturday 28 November** (closes of Friday 27 November) and saved as values in `Data/r3_bloomberg_values.xlsx`. Until that file comes from our screen, the notebook stops at §2 and the CSVs of §2–§3 are not overwritten.

### Input files

| File | Where it comes from | What it contains |
|---|---|---|
| `../Data/Backtest/AMC R3 BABQM BT.xlsx` | EQBT, Export to Excel | Monthly returns of BABQM and of the Russell 1000, Nov 2006 – Oct 2026 |
| `../Data/Backtest/AMC R3 BAB BT.xlsx` | EQBT | The low-beta rank alone: the same rule with one factor |
| `../Data/Tested Strategies.xlsx` | Our trials log | Every screen we backtested (Class 4: count the trials) |
| `Data/r3_bloomberg_values.xlsx`, sheet **Screen** | `BEQS` + `BDP` | Today's stocks of the screen, with name, GICS sector and last price |
| sheet **Prices** | `BDH`, monthly | 60+ month-end total return indices of the screen stocks |
| sheet **Other** | `BDH`, monthly | Our SAA ETFs, the Round 2 stocks, the 1-month T-bill (`GB1M Index`) |
| sheet **Live** | `BDP` | Last price of the SAA ETFs and of the Round 2 stocks |

### Output files

| CSV | Contents | Built from |
|---|---|---|
| `r3_backtest.csv` | Monthly backtest returns | EQBT files |
| `r3_trials.csv` | The backtests we tried | Trials log |
| `r3_holdings.csv` | Stocks in the portfolio, with last price | Screen, Live |
| `r3_prices.csv` | Monthly total return index of all assets | Prices, Other |
| `r3_rf.csv` | Monthly risk-free rate | Other (T-bill) |

### Round 3 in one box (full recap at the top of notebook 02)
- **Task:** add a **quantitative strategy** (Class 6) and a macro strategy (Class 7) on top of the **unchanged SAA**. **Long-only**, stocks and ETFs. Keep or close each Round 2 stock, with a reason.
- **Risk budget:** ex-ante **tracking error vs the SAA ≤ 7%** for all deviations together (monthly USD returns, last 60 months, annualized). Report volatility and 1-month 5% VaR next to the SAA's.
- **Checkpoint 1 (quant):** factor and rationale → construction rule → Class 4 backtest (design, validation, risk, discount).
- **Dates:** trades in PRTU on 28–29 November with the closes of **Friday 27 November**; report by Sunday 29 November, 23:59.
- **What this notebook does:** only data. Notebook 02 uses every number for one of the points above.

In [1]:
import numpy as np
import pandas as pd
pd.set_option("display.float_format", "{:.4f}".format)

SCREEN = "AMC R3 BABQM"                            # our EQS screen: must match Settings!B4 of the Bloomberg file
DATA   = "Data/"                                   # Bloomberg files of the terminal refresh
FILE   = DATA + "r3_bloomberg_values.xlsx"         # r3_bloomberg.xlsx after the refresh, saved as values (Paste Values)
BT_DIR = "../Data/Backtest/"                       # our EQBT exports
EQBT   = {"BABQM": "AMC R3 BABQM BT.xlsx",         # strategy name -> EQBT file; the first one is the strategy we invest in
          "BAB":   "AMC R3 BAB BT.xlsx"}           # its low-beta rank alone (add quality and momentum alone here if we run them)
TRIALS = "../Data/Tested Strategies.xlsx"          # trials log: one row per backtest we ran
SAA    = ["VTI", "VEA", "IEMG", "BND", "BNDX", "TIP", "GLD", "BCI"]    # Round 1 SAA ETFs
ROUND2 = ["NVDA", "PEP", "ABBV", "DRI"]            # Round 2 stocks (all kept, at 2% each: W_R2 in notebook 02)

def to_dates(d):
    # Excel dates can arrive as dates or as serial numbers (days since 1899-12-30)
    if pd.api.types.is_datetime64_any_dtype(d):
        return d
    num = pd.to_numeric(d, errors="coerce")
    if num.notna().sum() > d.notna().sum() / 2:                        # mostly numbers -> Excel serial dates
        return pd.to_datetime(num, unit="D", origin="1899-12-30", errors="coerce")
    return pd.to_datetime(d, errors="coerce", format="mixed")

def read_bdh(sheet):
    # One BDH per ticker: row 1 = ticker, then two columns (date, value) per ticker
    raw = pd.read_excel(FILE, sheet_name=sheet, header=None)
    out = {}
    for j in range(0, raw.shape[1] - 1, 2):
        t = raw.iat[0, j]
        if not isinstance(t, str) or not t.strip():                    # empty slot
            continue
        s = pd.Series(pd.to_numeric(raw.iloc[1:, j + 1], errors="coerce").values,
                      index=to_dates(raw.iloc[1:, j]).values).dropna()  # '#N/A ...' rows are dropped
        s = s[s.index.notna()]
        if len(s):
            out[t.replace(" US Equity", "").replace(" Equity", "")] = s
    df = pd.DataFrame(out)
    df.index = pd.DatetimeIndex(df.index) + pd.offsets.MonthEnd(0)     # label = calendar month-end
    return df.groupby(level=0).last().sort_index()

## 1. The EQS backtest
EQBT applies the screen **at each rebalancing date with the data of that date** and the **historical** members of the Russell 1000: no look-ahead and no survivorship bias (the two avoidable biases of Class 4).

Each export has four columns: `Rebalance Date` (mm/dd/yyyy), the **periodic return** of the screen (%), its turnover (%) and the periodic return of the **benchmark** (RIY = Russell 1000). The first row is the start date (all zeros): we drop it.

In [2]:
def read_eqbt(file):
    # one EQBT export -> DataFrame with the screen return, its turnover, the benchmark return (decimals)
    x = pd.read_excel(BT_DIR + file)
    if "RIY" not in str(x.columns[3]):
        raise ValueError(f"{file}: the benchmark is '{x.columns[3]}', not the Russell 1000 (RIY)")
    x.index = pd.to_datetime(x.iloc[:, 0], format="%m/%d/%Y") + pd.offsets.MonthEnd(0)   # label = calendar month-end
    x = x.iloc[:, 1:4].apply(pd.to_numeric, errors="coerce") / 100                       # % -> decimals
    x.columns = ["ret", "turnover", "bench"]
    return x.iloc[1:]                                                                     # drop the start row (all 0)

raw = {k: read_eqbt(f) for k, f in EQBT.items()}
STRAT = next(iter(EQBT))
bt = pd.DataFrame({k: x["ret"] for k, x in raw.items()})
bt["Russell 1000"] = raw[STRAT]["bench"]

# checks: same months and same benchmark in every file, one row per month, no gaps, no missing values
same_bench = all(x.index.equals(bt.index) and np.allclose(x["bench"], bt["Russell 1000"]) for x in raw.values())
gaps = (bt.index.to_series().diff().dt.days > 31).sum()
print(f"{bt.index[0]:%b %Y} -> {bt.index[-1]:%b %Y} | {len(bt)} months | gaps: {gaps} | "
      f"missing: {bt.isna().sum().sum()} | same benchmark in all files: {same_bench}")
print(f"average monthly turnover of {STRAT}: {raw[STRAT]['turnover'].mean():.1%} of the portfolio")

bt.to_csv("r3_backtest.csv")          # saved now: notebook 02 §3-§5 can run before the Bloomberg refresh
bt.describe().T[["count", "mean", "std", "min", "max"]]

Nov 2006 -> Oct 2026 | 240 months | gaps: 0 | missing: 0 | same benchmark in all files: True
average monthly turnover of BABQM: 26.7% of the portfolio


,count,mean,std,min,max
BABQM,240.0000,0.0079,0.0411,-0.1457,0.1213
BAB,240.0000,0.0066,0.0400,-0.1472,0.2156
Russell 1000,240.0000,0.0097,0.0450,-0.1746,0.1321


### The trials log (Class 4)
Every screen we backtested is a trial: with 20 tries and no real effect, about one looks significant at 5%. `Tested Strategies.xlsx` lists them all. The Sharpe, IR and beta in it are Bloomberg's, from the EQBT summary.

In [3]:
log = pd.read_excel(TRIALS, header=3).dropna(subset=["Strategy Name"])     # headers on row 4
trials = log.set_index("Strategy Name")[["Category", "Universe", "# Holdings", "Rebalance", "Benchmark",
                                         "Total Return", "Sharpe Ratio", "Info Ratio", "Beta", "Status"]]
trials.to_csv("r3_trials.csv")
print(f"backtests in the trials log: {len(trials)}")
trials

backtests in the trials log: 9


,Category,Universe,# Holdings,Rebalance,Benchmark,Total Return,Sharpe Ratio,Info Ratio,Beta,Status
Strategy Name,,,,,,,,,,
AMC R3 BABQM,Multi-Factor,Russell 1000 (active),30,Monthly,RIY,4.3803,0.5300,-0.2200,0.6600,Tested
MCM Momentun Nov 26,Multi-Factor,S&P 500 + Morningstar US Total Market + FTSE D...,30,Monthly,RIY,8.1202,0.7700,0.2100,1.0600,Promising
AMC R3 MQE,Multi-Factor,Russell 1000 (active),30,Monthly,RIY,7.8026,0.5500,0.1200,1.0400,Tested
"AMC R3 SVQ (Quarterly, vs RIY)",Multi-Factor,Russell 2000 (active),35,Quarterly,RIY,1.9274,0.2900,-0.1200,1.2100,Tested
AMC R3 MWC,Multi-Factor,Russell 1000 (active),30,Monthly,RIY,6.7023,0.4900,0.1000,1.1300,Tested
AMC R3 SVQ,Multi-Factor,Russell 2000 (active),35,Monthly,RTY,3.0200,0.3500,0.0500,1.0900,Tested
AMC R3 BAB,Low Vol,Russell 1000 (active),31,Monthly,RIY,3.0127,0.4800,-0.2900,0.4500,Tested
AMC R3 VMQ,Multi-Factor,Russell 1000 (active),28,Monthly,RIY,5.8248,0.4900,0.0200,1.0600,Tested
AMC R3 SCV,Value,Russell 2000 (active),55,Monthly,RTY,1.3287,0.3000,0.1100,1.4200,Tested


## 2. Today's portfolio: the stocks of the screen
`BEQS` returns the stocks that pass the screen **today** (in the challenge: refresh on Saturday 28 November → data of Friday 27 November).

⚠️ **Wait until every `#N/A Requesting Data...` has disappeared before you copy and paste values.**

The first cell checks that the file was refreshed with **our** screen. The TA's example file has the same name, so without this check the notebook would build the portfolio from the TA's stocks.

In [4]:
import os
if not os.path.exists(FILE):
    raise FileNotFoundError(f"{FILE} is not there yet. Refresh 'Bloomberg template/r3_bloomberg.xlsx' on the terminal "
                            "(README, step 2), save it as values under this name, then run this notebook again. "
                            "§1 is already saved.")
settings = pd.read_excel(FILE, sheet_name="Settings", header=None, index_col=0)[1]
if settings.get("Screen name") != SCREEN:
    raise ValueError(f"{FILE} comes from the screen '{settings.get('Screen name')}', not '{SCREEN}'. Refresh "
                     "'Bloomberg template/r3_bloomberg.xlsx' on the terminal (README, step 2), save it as values "
                     "under this name, then run this notebook again. §1 is already saved.")
END = pd.Timestamp(settings["End date"])
print(f"Bloomberg file of the screen '{SCREEN}', prices to {END:%a %d %b %Y}")
if END != pd.Timestamp("2026-11-27"):
    print("⚠️ the Round 3 trades use the closes of Friday 27 November 2026: check Settings!B3")

Bloomberg file of the screen 'AMC R3 BABQM', prices to Fri 27 Nov 2026


We read the screen in two blocks:
- **`BEQS` (columns A:J):** the ticker, the short name, then the columns of our screen. We find the score and the market cap by their header, since the columns depend on how the screen was saved.
- **`BDP` (columns L:O):** Bloomberg ticker, name, GICS sector and last price of each stock. They are the same for any screen.

In [5]:
scr_raw = pd.read_excel(FILE, sheet_name="Screen", header=None).reindex(columns=range(17))
head = scr_raw.iloc[1, :11].astype(str).str.lower()                  # BEQS headers (row 2)
body = scr_raw.iloc[2:]
body = body[body[0].astype(str).str.endswith(" Equity")]             # drop empty and '#N/A' rows

def num(s):                                                           # '7’372’253’953.98' -> 7372253953.98
    return pd.to_numeric(s.astype(str).str.replace("’", "").str.replace("'", ""), errors="coerce")

def beqs(*words):                                                     # the BEQS column whose header has the first word found
    for w in words:
        for j, h in head.items():
            if w in h:
                return num(body[j])
    return pd.Series(np.nan, index=body.index)

def loaded(s):                                                        # a BDP cell that loaded (not empty, not '#N/A ...')
    return s.notna() & ~s.astype(str).str.startswith("#N/A")

scr = pd.DataFrame({"ticker":  body[0].str.replace(" US Equity", "").str.replace(" Equity", ""),
                    "name":    body[12].where(loaded(body[12]), body[1]),     # BDP name, else the BEQS short name
                    "sector":  body[13].where(loaded(body[13])),              # BDP GICS_SECTOR_NAME
                    "px_last": num(body[14]),                                 # BDP PX_LAST
                    "score":   beqs("babqm", "score", "rank"),
                    "mkt_cap": beqs("market cap")})

SECTOR_BY_HAND = {}       # ⚠️ only for a sector that did not load: "TICKER": "GICS sector" (check with DES <GO>)
scr["sector"] = scr["sector"].fillna(scr["ticker"].map(SECTOR_BY_HAND)).fillna("?")

live = pd.read_excel(FILE, sheet_name="Live").iloc[:, :3]             # SAA ETFs and Round 2 stocks
live.columns = ["ticker", "name", "px_last"]
live = live.dropna(subset=["ticker"])
live["ticker"] = live["ticker"].str.replace(" US Equity", "")
live["px_last"] = pd.to_numeric(live["px_last"], errors="coerce")
missing = sorted(set(SAA + ROUND2) - set(live.loc[live["px_last"].notna(), "ticker"]))
if missing:
    raise ValueError(f"sheet Live has no price for {missing}: add them to the template and refresh")

both = sorted(set(scr["ticker"]) & set(ROUND2))                       # a Round 2 stock that the screen also picks
if both:
    print(f"⚠️ in the screen and in Round 2: {both}. Kept as one position: the quant weight adds to the Round 2 one (notebook 02 §8)")
holdings = pd.concat([scr[["ticker", "name", "sector", "px_last", "score", "mkt_cap"]].assign(group="Quant"),
                      live[~live["ticker"].isin(scr["ticker"])].assign(group="SAA / Round 2", sector="")]).set_index("ticker")

# checks: about 30 stocks (the trials log says 30), each with a price and a sector; score >= 98 (the filter of the screen)
print(f"stocks in the screen today: {len(scr)} | missing price: {scr['px_last'].isna().sum()} | "
      f"unknown sector: {(scr['sector'] == '?').sum()} | score range: {scr['score'].min():.0f}-{scr['score'].max():.0f}")
if scr["px_last"].isna().any():
    print("⚠️ stocks without a price:", scr.loc[scr["px_last"].isna(), "ticker"].tolist())
display(scr["sector"].value_counts().to_frame("stocks"))
holdings

stocks in the screen today: 29 | missing price: 0 | unknown sector: 0 | score range: 98-100


,stocks
sector,
Energy,9
Health Care,6
Consumer Staples,6
Industrials,3
Consumer Discretionary,2
Materials,1
Financials,1
Information Technology,1


,name,sector,px_last,score,mkt_cap,group
ticker,,,,,,
MPC,MARATHON PETROLEUM CORP,Energy,359.8000,100.0000,97967161020.2400,Quant
MUSA,MURPHY USA INC,Consumer Discretionary,510.0200,100.0000,9231317429.4100,Quant
HALO,HALOZYME THERAPEUTICS INC,Health Care,94.8000,100.0000,10808148347.9300,Quant
PM,PHILIP MORRIS INTERNATIONAL,Consumer Staples,187.8000,100.0000,292715531882.8400,Quant
MO,ALTRIA GROUP INC,Consumer Staples,76.8900,100.0000,127901663379.0700,Quant
HRB,H&R BLOCK INC,Consumer Discretionary,47.1500,100.0000,5731228520.4300,Quant
VLO,VALERO ENERGY CORP,Energy,329.3900,100.0000,91861736601.2200,Quant
KO,COCA-COLA CO/THE,Consumer Staples,97.4000,100.0000,419005293993.1800,Quant
CF,CF INDUSTRIES HOLDINGS INC,Materials,119.5000,100.0000,17481046864.5000,Quant


## 3. Monthly returns for the tracking error
The **total return index** reinvests dividends, as PRTU does with our settings: $R_{i,t} = TR_{i,t}/TR_{i,t-1} - 1$. Same convention as Rounds 1 and 2: **monthly returns in USD, last 60 months**. The T-bill (`GB1M Index`, % per year) becomes a monthly rate $r_f = \text{yield}/100/12$.

In [6]:
P_stocks = read_bdh("Prices")
P_other  = read_bdh("Other")
missing = sorted(set(SAA + ROUND2 + ["GB1M Index"]) - set(P_other.columns))
if missing:
    raise ValueError(f"sheet Other has no prices for {missing}: add them to the template and refresh")
no_hist = sorted(set(scr["ticker"]) - set(P_stocks.columns))
if no_hist:
    print(f"⚠️ screen stocks without a price history in sheet Prices: {no_hist}")

rf = (P_other.pop("GB1M Index").dropna() / 100 / 12).rename("rf")    # monthly T-bill rate, from 2006 (backtest)
P_stocks = P_stocks.drop(columns=[c for c in P_stocks if c in P_other])   # a Round 2 stock that the screen also picks
prices = pd.concat([P_other, P_stocks], axis=1).loc[P_stocks.index[0]:]

# checks: 61 month-ends (60 returns); stocks with a shorter history (recent IPOs) are fine, but note them
n_obs = prices.notna().sum()
print(f"{prices.index[0]:%b %Y} -> {prices.index[-1]:%b %Y} | {prices.shape[1]} series | "
      f"shorter than {len(prices)} months: {n_obs[n_obs < len(prices)].to_dict()}")
print(f"T-bill: {rf.index[0]:%b %Y} -> {rf.index[-1]:%b %Y}, last {rf.iloc[-1] * 1200:.2f}% per year")
prices.pct_change(fill_method=None).describe().T[["count", "mean", "std", "min", "max"]].head(12)

Sep 2021 -> Oct 2026 | 41 series | shorter than 62 months: {}
T-bill: Jan 2006 -> Oct 2026, last 3.96% per year


,count,mean,std,min,max
VTI,61.0000,0.0107,0.0457,-0.0922,0.1040
VEA,61.0000,0.0088,0.0465,-0.0986,0.1255
IEMG,61.0000,0.0067,0.0505,-0.1131,0.1466
BND,61.0000,0.0000,0.0187,-0.0418,0.0454
BNDX,61.0000,0.0003,0.0154,-0.0361,0.0343
TIP,61.0000,0.0004,0.0178,-0.0672,0.0430
GLD,61.0000,0.0140,0.0503,-0.1168,0.1227
BCI,61.0000,0.0092,0.0445,-0.1072,0.1137
NVDA,61.0000,0.0484,0.1447,-0.3203,0.3634
PEP,61.0000,0.0023,0.0485,-0.0958,0.1122


In [7]:
holdings.to_csv("r3_holdings.csv")    # today's stocks + ETFs + Round 2 stocks: name, sector, last price
prices.to_csv("r3_prices.csv")        # monthly total return index (TE)
rf.to_csv("r3_rf.csv")                # monthly T-bill rate
print("saved: r3_holdings.csv, r3_prices.csv, r3_rf.csv (r3_backtest.csv and r3_trials.csv in §1)")

saved: r3_holdings.csv, r3_prices.csv, r3_rf.csv (r3_backtest.csv and r3_trials.csv in §1)
